**Imports**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

**Generated Dummy Data**

In [ ]:
np.random.seed(42)

n_samples = 500
true_w, true_b = 3.5, 7.0          # asal weight/bias jo hum recover karna chahte hain

X = np.random.rand(n_samples, 1) * 10          # X values 0-10 ke beech
noise = np.random.randn(n_samples, 1) * 1.5     # thora sa random noise
y = true_w * X + true_b + noise                 # y = 3.5x + 7 + noise

print(f"Dummy data ready: {n_samples} samples")
print(f"True parameters -> w = {true_w}, b = {true_b}")

# Dekh lein data ka preview
df = pd.DataFrame({'X': X.flatten(), 'y': y.flatten()})
df.head(10)

**Loss, Gradient & Accuracy (R²) Functions**

In [ ]:
def compute_loss(X, y, w, b):
    y_pred = w * X + b
    return np.mean((y_pred - y) ** 2)


def compute_gradients(X_batch, y_batch, w, b):
    n = len(X_batch)
    y_pred = w * X_batch + b
    error = y_pred - y_batch
    dw = (2 / n) * np.sum(error * X_batch)
    db = (2 / n) * np.sum(error)
    return dw, db


def compute_accuracy(X, y, w, b):
    """R² score used as the 'accuracy' metric for this regression problem.
    R² = 1 -> perfect fit, R² = 0 -> model no better than predicting the mean."""
    y_pred = w * X + b
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - (ss_res / ss_tot)

**Batch Gradient Descent**

In [ ]:
def batch_gradient_descent(X, y, lr=0.01, epochs=100):
    w, b = 0.0, 0.0
    losses = []
    accuracies = []
    for epoch in range(epochs):
        dw, db = compute_gradients(X, y, w, b)
        w -= lr * dw
        b -= lr * db
        losses.append(compute_loss(X, y, w, b))
        accuracies.append(compute_accuracy(X, y, w, b))
    return w, b, losses, accuracies

**Stochastic Gradient Descent (SGD)**

In [ ]:
def stochastic_gradient_descent(X, y, lr=0.01, epochs=100):
    w, b = 0.0, 0.0
    losses = []
    accuracies = []
    n = len(X)
    for epoch in range(epochs):
        indices = np.random.permutation(n)   # data shuffle har epoch
        for i in indices:
            X_i = X[i:i+1]
            y_i = y[i:i+1]
            dw, db = compute_gradients(X_i, y_i, w, b)
            w -= lr * dw
            b -= lr * db
        losses.append(compute_loss(X, y, w, b))          # epoch ke end pe pure data ka loss
        accuracies.append(compute_accuracy(X, y, w, b))  # epoch ke end pe pure data ka R²
    return w, b, losses, accuracies

**Mini-batch Gradient Descent**

In [ ]:
def minibatch_gradient_descent(X, y, lr=0.01, epochs=100, batch_size=32):
    w, b = 0.0, 0.0
    losses = []
    accuracies = []
    n = len(X)
    for epoch in range(epochs):
        indices = np.random.permutation(n)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        for start in range(0, n, batch_size):
            end = start + batch_size
            X_batch = X_shuffled[start:end]
            y_batch = y_shuffled[start:end]
            dw, db = compute_gradients(X_batch, y_batch, w, b)
            w -= lr * dw
            b -= lr * db
        losses.append(compute_loss(X, y, w, b))
        accuracies.append(compute_accuracy(X, y, w, b))
    return w, b, losses, accuracies

**Adam Optimizer**

In [ ]:
def adam_optimizer(X, y, lr=0.01, epochs=100, batch_size=32,
                    beta1=0.9, beta2=0.999, eps=1e-8):
    w, b = 0.0, 0.0
    m_w, v_w = 0.0, 0.0     # w ke liye moment estimates
    m_b, v_b = 0.0, 0.0     # b ke liye moment estimates
    losses = []
    accuracies = []
    n = len(X)
    t = 0   # timestep counter (bias correction ke liye)

    for epoch in range(epochs):
        indices = np.random.permutation(n)
        X_shuffled = X[indices]
        y_shuffled = y[indices]

        for start in range(0, n, batch_size):
            end = start + batch_size
            X_batch = X_shuffled[start:end]
            y_batch = y_shuffled[start:end]

            dw, db = compute_gradients(X_batch, y_batch, w, b)
            t += 1

            # Momentum update (1st moment)
            m_w = beta1 * m_w + (1 - beta1) * dw
            m_b = beta1 * m_b + (1 - beta1) * db

            # RMSProp-style update (2nd moment)
            v_w = beta2 * v_w + (1 - beta2) * (dw ** 2)
            v_b = beta2 * v_b + (1 - beta2) * (db ** 2)

            # Bias correction
            m_w_hat = m_w / (1 - beta1 ** t)
            m_b_hat = m_b / (1 - beta1 ** t)
            v_w_hat = v_w / (1 - beta2 ** t)
            v_b_hat = v_b / (1 - beta2 ** t)

            # Parameter update
            w -= lr * m_w_hat / (np.sqrt(v_w_hat) + eps)
            b -= lr * m_b_hat / (np.sqrt(v_b_hat) + eps)

        losses.append(compute_loss(X, y, w, b))
        accuracies.append(compute_accuracy(X, y, w, b))
    return w, b, losses, accuracies

**Run all techniques**

Har configuration ke liye 2 graphs banenge: **Loss curve** (MSE kam ho raha hai ya nahi) aur
**Accuracy curve** (R² score — 1.0 ke jitna qareeb, model utna behtar fit ho raha hai).

### Config 1: Epochs=300, LR=0.01, Batch=16

In [ ]:
EPOCHS = 300
LR = 0.01

w_bgd, b_bgd, loss_bgd, acc_bgd = batch_gradient_descent(X, y, lr=LR, epochs=EPOCHS)
w_sgd, b_sgd, loss_sgd, acc_sgd = stochastic_gradient_descent(X, y, lr=LR, epochs=EPOCHS)
w_mbgd, b_mbgd, loss_mbgd, acc_mbgd = minibatch_gradient_descent(X, y, lr=LR, epochs=EPOCHS, batch_size=16)
w_adam, b_adam, loss_adam, acc_adam = adam_optimizer(X, y, lr=LR, epochs=EPOCHS, batch_size=16)

print("--- Final Learned Parameters ---")
print(f"Batch GD      -> w = {w_bgd:.4f}, b = {b_bgd:.4f}, final loss = {loss_bgd[-1]:.4f}, final accuracy (R2) = {acc_bgd[-1]:.4f}")
print(f"SGD           -> w = {w_sgd:.4f}, b = {b_sgd:.4f}, final loss = {loss_sgd[-1]:.4f}, final accuracy (R2) = {acc_sgd[-1]:.4f}")
print(f"Mini-batch GD -> w = {w_mbgd:.4f}, b = {b_mbgd:.4f}, final loss = {loss_mbgd[-1]:.4f}, final accuracy (R2) = {acc_mbgd[-1]:.4f}")
print(f"Adam          -> w = {w_adam:.4f}, b = {b_adam:.4f}, final loss = {loss_adam[-1]:.4f}, final accuracy (R2) = {acc_adam[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss_bgd, label="Batch GD", linewidth=2)
axes[0].plot(loss_sgd, label="Stochastic GD", linewidth=2)
axes[0].plot(loss_mbgd, label="Mini-batch GD", linewidth=2)
axes[0].plot(loss_adam, label="Adam", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (MSE)")
axes[0].set_title("Loss Curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(acc_bgd, label="Batch GD", linewidth=2)
axes[1].plot(acc_sgd, label="Stochastic GD", linewidth=2)
axes[1].plot(acc_mbgd, label="Mini-batch GD", linewidth=2)
axes[1].plot(acc_adam, label="Adam", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (R\u00b2 Score)")
axes[1].set_title("Accuracy Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Config 1: Epochs=300, LR=0.01, Batch=16")
plt.tight_layout()
plt.show()

### Config 2: Epochs=250, LR=0.001, Batch=16

In [ ]:
EPOCHS = 250
LR = 0.001

w_bgd, b_bgd, loss_bgd, acc_bgd = batch_gradient_descent(X, y, lr=LR, epochs=EPOCHS)
w_sgd, b_sgd, loss_sgd, acc_sgd = stochastic_gradient_descent(X, y, lr=LR, epochs=EPOCHS)
w_mbgd, b_mbgd, loss_mbgd, acc_mbgd = minibatch_gradient_descent(X, y, lr=LR, epochs=EPOCHS, batch_size=16)
w_adam, b_adam, loss_adam, acc_adam = adam_optimizer(X, y, lr=LR, epochs=EPOCHS, batch_size=16)

print("--- Final Learned Parameters ---")
print(f"Batch GD      -> w = {w_bgd:.4f}, b = {b_bgd:.4f}, final loss = {loss_bgd[-1]:.4f}, final accuracy (R2) = {acc_bgd[-1]:.4f}")
print(f"SGD           -> w = {w_sgd:.4f}, b = {b_sgd:.4f}, final loss = {loss_sgd[-1]:.4f}, final accuracy (R2) = {acc_sgd[-1]:.4f}")
print(f"Mini-batch GD -> w = {w_mbgd:.4f}, b = {b_mbgd:.4f}, final loss = {loss_mbgd[-1]:.4f}, final accuracy (R2) = {acc_mbgd[-1]:.4f}")
print(f"Adam          -> w = {w_adam:.4f}, b = {b_adam:.4f}, final loss = {loss_adam[-1]:.4f}, final accuracy (R2) = {acc_adam[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss_bgd, label="Batch GD", linewidth=2)
axes[0].plot(loss_sgd, label="Stochastic GD", linewidth=2)
axes[0].plot(loss_mbgd, label="Mini-batch GD", linewidth=2)
axes[0].plot(loss_adam, label="Adam", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (MSE)")
axes[0].set_title("Loss Curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(acc_bgd, label="Batch GD", linewidth=2)
axes[1].plot(acc_sgd, label="Stochastic GD", linewidth=2)
axes[1].plot(acc_mbgd, label="Mini-batch GD", linewidth=2)
axes[1].plot(acc_adam, label="Adam", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (R\u00b2 Score)")
axes[1].set_title("Accuracy Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Config 2: Epochs=250, LR=0.001, Batch=16")
plt.tight_layout()
plt.show()

### Config 3: Epochs=500, LR=0.0001, Batch=16

In [ ]:
EPOCHS = 500
LR = 0.0001

w_bgd, b_bgd, loss_bgd, acc_bgd = batch_gradient_descent(X, y, lr=LR, epochs=EPOCHS)
w_sgd, b_sgd, loss_sgd, acc_sgd = stochastic_gradient_descent(X, y, lr=LR, epochs=EPOCHS)
w_mbgd, b_mbgd, loss_mbgd, acc_mbgd = minibatch_gradient_descent(X, y, lr=LR, epochs=EPOCHS, batch_size=16)
w_adam, b_adam, loss_adam, acc_adam = adam_optimizer(X, y, lr=LR, epochs=EPOCHS, batch_size=16)

print("--- Final Learned Parameters ---")
print(f"Batch GD      -> w = {w_bgd:.4f}, b = {b_bgd:.4f}, final loss = {loss_bgd[-1]:.4f}, final accuracy (R2) = {acc_bgd[-1]:.4f}")
print(f"SGD           -> w = {w_sgd:.4f}, b = {b_sgd:.4f}, final loss = {loss_sgd[-1]:.4f}, final accuracy (R2) = {acc_sgd[-1]:.4f}")
print(f"Mini-batch GD -> w = {w_mbgd:.4f}, b = {b_mbgd:.4f}, final loss = {loss_mbgd[-1]:.4f}, final accuracy (R2) = {acc_mbgd[-1]:.4f}")
print(f"Adam          -> w = {w_adam:.4f}, b = {b_adam:.4f}, final loss = {loss_adam[-1]:.4f}, final accuracy (R2) = {acc_adam[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss_bgd, label="Batch GD", linewidth=2)
axes[0].plot(loss_sgd, label="Stochastic GD", linewidth=2)
axes[0].plot(loss_mbgd, label="Mini-batch GD", linewidth=2)
axes[0].plot(loss_adam, label="Adam", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (MSE)")
axes[0].set_title("Loss Curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(acc_bgd, label="Batch GD", linewidth=2)
axes[1].plot(acc_sgd, label="Stochastic GD", linewidth=2)
axes[1].plot(acc_mbgd, label="Mini-batch GD", linewidth=2)
axes[1].plot(acc_adam, label="Adam", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (R\u00b2 Score)")
axes[1].set_title("Accuracy Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Config 3: Epochs=500, LR=0.0001, Batch=16")
plt.tight_layout()
plt.show()

**Comparison Across Hyperparameter Configurations**

In [ ]:
# Different hyperparameter configurations to compare
configs = [
    {"lr": 0.01,   "epochs": 300, "batch_size": 32, "title": "LR=0.01, Epochs=300, Batch=32"},
    {"lr": 0.01,   "epochs": 300, "batch_size": 16, "title": "LR=0.01, Epochs=300, Batch=16"},
    {"lr": 0.001,  "epochs": 250, "batch_size": 32, "title": "LR=0.001, Epochs=250, Batch=32"},
    {"lr": 0.0001, "epochs": 500, "batch_size": 16, "title": "LR=0.0001, Epochs=500, Batch=16"},
]

results = []
for cfg in configs:
    lr, epochs, bs = cfg["lr"], cfg["epochs"], cfg["batch_size"]
    _, _, l_bgd, a_bgd   = batch_gradient_descent(X, y, lr=lr, epochs=epochs)
    _, _, l_sgd, a_sgd   = stochastic_gradient_descent(X, y, lr=lr, epochs=epochs)
    _, _, l_mb, a_mb     = minibatch_gradient_descent(X, y, lr=lr, epochs=epochs, batch_size=bs)
    _, _, l_adam, a_adam = adam_optimizer(X, y, lr=lr, epochs=epochs, batch_size=bs)
    results.append((cfg["title"], l_bgd, l_sgd, l_mb, l_adam, a_bgd, a_sgd, a_mb, a_adam))

print("Configurations run:", len(results))

*Loss comparison — all configurations*

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (title, l_bgd, l_sgd, l_mb, l_adam, *_ ) in zip(axes, results):
    ax.plot(l_bgd, label="Batch GD", linewidth=1.8)
    ax.plot(l_sgd, label="SGD", linewidth=1.8)
    ax.plot(l_mb, label="Mini-batch GD", linewidth=1.8)
    ax.plot(l_adam, label="Adam", linewidth=1.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (MSE)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle("Loss Comparison Across Different Hyperparameters", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

*Accuracy (R²) comparison — all configurations*

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (title, *_rest) in zip(axes, results):
    _, _, _, _, a_bgd, a_sgd, a_mb, a_adam = _rest
    ax.plot(a_bgd, label="Batch GD", linewidth=1.8)
    ax.plot(a_sgd, label="SGD", linewidth=1.8)
    ax.plot(a_mb, label="Mini-batch GD", linewidth=1.8)
    ax.plot(a_adam, label="Adam", linewidth=1.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (R² Score)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle("Accuracy (R²) Comparison Across Different Hyperparameters", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()